# 1. Data Acquisition

In [ ]:
import dxpy
import dxdata
import pandas as pd
import pyspark
import numpy as np

In [ ]:
# Auto-discover the assigned dataset ID and load the dataset
dispensed_dataset_id = dxpy.find_one_data_object(typename='Dataset', name='app*.dataset', folder='/', name_mode='glob')['id']
dataset = dxdata.load_dataset(id=dispensed_dataset_id)
dataset 

In [ ]:
# Access the participant entity
participant = dataset['participant']

# 2. Phenotype Extraction (All Instances)

In [ ]:
# Define field_names_for_ids to retrieve all field names for a list of field IDs
# UKB phenotype format: "p<FieldID>_i<instance>_a<array>" 
# e.g. p21001_i0_a0 for field ID 21001 (BMI), instance 0, array 0

def field_names_for_ids(field_ids):
    from distutils.version import LooseVersion 
    fields = []
    for field_id in field_ids:
        field_id = 'p' + str(field_id)
        fields += participant.find_fields(name_regex=r'^{}(_i\d+)?(_a\d+)?$'.format(field_id))
    return sorted([field.name for field in fields], key=lambda n: LooseVersion(n))


# Define field IDs to extract — adiposity-related traits

field_ids = [

    # body impedance
    '23099','23100','21001','48','49','23105',

    # MRI fat imaging
    '22407','22408','22432','22410','22415','22434',

    # DXA fat
    '23245','23249','23253','23257','23262',
    '23266','23270','23274','23278','23284',
    '23288','23289',

    # muscle
    '46','47','22436','24353','24354',
    '22435','23355','23356',

    # population
    '31','22001','21000','20116','21022','22009',

    # blood pressure
    '4080','4079',

    # lipids
    '30690','30760','30780','30870',

    # glucose
    '30740','30750',

    # inflammatory
    '30710',

    # renal
    '30700',

    # hepatic
    '30620',

]

field_names = ['eid'] + field_names_for_ids(field_ids)

In [ ]:
# Connect to Spark and convert to Pandas DataFrame
df = participant.retrieve_fields(names=field_names, engine=dxdata.connect())
pdf = df.toPandas()

In [ ]:
# Check column names
pdf.columns

In [ ]:
# Check dimensions (rows, columns)
pdf.shape

# 3. Phenotype Extraction (Baseline Only)

In [ ]:
# Define field_names_for_ids_baseline to retrieve only baseline (instance 0) field names
# Matches: pXXXX or pXXXX_i0 or pXXXX_i0_a*

def field_names_for_ids_baseline(field_ids):
    from distutils.version import LooseVersion
    fields = []
    for field_id in field_ids:
        field_id = 'p' + str(field_id)
        fields += participant.find_fields(
            name_regex=rf'^{field_id}(_i0)?(_a\d+)?$'
        )
    return sorted([f.name for f in fields], key=LooseVersion)
# Note: for some fields, the first measurement is not i0!
field_ids = []
baseline_field_names = ['eid'] + field_names_for_ids_baseline(field_ids)


In [ ]:
# Connect to Spark and convert to Pandas DataFrame
df_baseline = participant.retrieve_fields(names=baseline_field_names, engine=dxdata.connect())
pdf_baseline = df_baseline.toPandas()

# 4. Rename Field IDs to Readable Column Names

In [ ]:
# Rename columns from UKB field IDs to readable names
import re

field_map = {
    
    # body impedance measurement
    '23099': 'whole_body_fat_percentage',
    '23100': 'whole_body_fat_mass',
    '21001': 'BMI',
    '48': 'waist_circumference',
    '49': 'hip_circumference',
    '23105': 'basal_metabolic_rate',
    
    # MRI fat imaging
    '22407': 'VAT',
    '22408': 'ASAT',
    '22432': 'total_abdominal_adipose_index',
    '22410': 'total_trunk_fat_volume',
    '22415': 'total_adipose_tissue_volume',
    '22434': 'abdominal_fat_ratio',
    
    # body fat composition by DXA 
    '23245': 'android_fat_mass',
    '23249': 'arm_fat_mass_left',
    '23253': 'arm_fat_mass_right',
    '23257': 'arms_fat_mass',
    '23262': 'gynoid_fat_mass',
    '23266': 'leg_fat_mass_left',
    '23270': 'leg_fat_mass_right',
    '23274': 'legs_fat_mass',
    '23278': 'total_fat_mass',
    '23284': 'trunk_fat_mass',
    '23288': 'VAT_mass',
    '23289': 'VAT_volume',
    
    # muscle
    '46': 'hand_grip_strength_left',
    '47': 'hand_grip_strength_right',
    '22436': '10P_Liver_PDFF',
    '24353': 'Anterior_thigh_MFI_left',
    '24354': 'Anterior_thigh_MFI_right',
    '22435': 'Muscle_fat_infiltration(MFI)',
    '23355': 'Posterior_thigh_MFI_left',
    '23356': 'Posterior_thigh_MFI_right',

    # population characteristic
    '31': 'sex',
    '22001': 'genetic_sex',
    '21000': 'ethnicity',
    '20116': 'smoking',
    '21022':'age_at_recruitment',
    '22009': 'genetic_PCs',
    
    # blood pressure
    '4080': 'SBP',
    '4079': 'DBP',

    # lipids
    '30690': 'TC',
    '30760': 'HDL_C',
    '30780': 'LDL_C',
    '30870': 'Triglyceride',

    # glucose
    '30740': 'glucose',
    '30750': 'HbA1c',

    # inflammatory
    '30710': 'CRP',
    
    # renal function
    '30700': 'Creatinine',
    
    # hepatic function
    '30620': 'Alanine_aminotransferase',
    

    
}



def rename_ukb_columns_keep_instance(df, field_map):
    new_cols = {}
    for col in df.columns:
        for fid, new_name in field_map.items():
            m = re.match(rf'^p{fid}(_i\d+)?(_a\d+)?$', col)
            if m:
                suffix = col.replace(f'p{fid}', '')
                new_cols[col] = new_name + suffix
                break
    return df.rename(columns=new_cols)



pdf_renamed = rename_ukb_columns_keep_instance(pdf, field_map)
pdf_renamed = pdf_renamed.rename(columns={'eid':'participant_ID'})
pdf_renamed.head(6)


In [ ]:
# Rename columns for baseline DataFrame
def rename_ukb_columns_baseline(df, field_map):
    new_cols = {}
    for col in df.columns:
        for fid, new_name in field_map.items():
            # Match three cases:
            # p21001
            # p21001_i0
            # p21001_i0_a0 
            m = re.match(rf'^p{fid}(_i0)?(_a\d+)?$', col)
            if m:
                # Keep array suffix (e.g. _a0, _a1)
                suffix = m.group(2) if m.group(2) else ''
                new_cols[col] = new_name + suffix
                break
    return df.rename(columns=new_cols)

pdf_baseline_renamed = rename_ukb_columns_baseline(pdf_baseline, field_map)
pdf_baseline_renamed = pdf_baseline_renamed.rename(columns={'eid':'participant_ID'})

In [ ]:
pdf_renamed.columns

# 5. Ethnicity Mapping

In [ ]:
import numpy as np

# Map ethnicity numeric codes to readable labels
# Top-level ethnicity groups
ethnicity_group_map = {
    1: 'White',
    2: 'Mixed',
    3: 'Asian or Asian British',
    4: 'Black or Black British',
    5: 'Chinese',
    6: 'Other ethnic group',
    -1: 'Do not know',
    -3: 'Prefer not to answer'
}

# Sub-level ethnicity categories
ethnicity_detail_map = {
    1001: 'British',
    1002: 'Irish',
    1003: 'Any other white background',

    2001: 'White and Black Caribbean',
    2002: 'White and Black African',
    2003: 'White and Asian',
    2004: 'Any other mixed background',

    3001: 'Indian',
    3002: 'Pakistani',
    3003: 'Bangladeshi',
    3004: 'Any other Asian background',

    4001: 'Caribbean',
    4002: 'African',
    4003: 'Any other Black background'
}

def parse_ethnicity(code):
    if pd.isna(code):
        return pd.Series([np.nan, np.nan])

    code = int(code)

    # Case 1: top-level code
    if code in ethnicity_group_map:
        group  = ethnicity_group_map[code]
        detail = group

    # Case 2: sub-level code, e.g. 1001 / 3002 / 4001
    else:
        group_code = code // 1000
        group  = ethnicity_group_map.get(group_code, 'Unknown')
        detail = ethnicity_detail_map.get(code, 'Unknown')

    return pd.Series([group, detail])

pdf_renamed[['ethnicity_group', 'ethnicity_detail']] = pdf_renamed['ethnicity_i0'].apply(parse_ethnicity)

pdf_renamed.head(6)

# 6. Sex Mapping

In [ ]:
sex_map = {
    0: 'Female',
    1: 'Male'
}

pdf_renamed['sex_label'] = pdf_renamed['sex'].map(sex_map)
pdf_renamed.head(6)

In [ ]:
pdf_renamed['genetic_sex'].head(6)

In [ ]:
g_sex_map = {
    0.0: 'Female',
    1.0: 'Male'
}

pdf_renamed['genetic_sex_label'] = pdf_renamed['genetic_sex'].map(g_sex_map)
pdf_renamed.head(6)

# 7. Sex vs Genetic Sex Mismatch

In [ ]:
mismatch = pdf_renamed[pdf_renamed['sex_label'] != pdf_renamed['genetic_sex_label']]
mismatch[['participant_ID','sex_label','genetic_sex_label']].head(15)

In [ ]:
len(mismatch)

In [ ]:
true_mismatch = pdf_renamed[
    pdf_renamed['sex_label'].notna() &
    pdf_renamed['genetic_sex_label'].notna() &
    (pdf_renamed['sex_label'] != pdf_renamed['genetic_sex_label'])
]

In [ ]:
true_mismatch[['participant_ID','sex_label','genetic_sex_label']].head(6)

In [ ]:
len(true_mismatch)

# 8. Remove Sex-Mismatch Participants

In [ ]:
pdf_renamed = pdf_renamed[~pdf_renamed['participant_ID'].isin(true_mismatch['participant_ID'])]

In [ ]:
len(pdf_renamed)

In [ ]:
# Check dimensions (rows, columns)
pdf_renamed.shape

In [ ]:
pdf_renamed.columns

In [ ]:
pdf_renamed.head(6)

In [ ]:
pdf_renamed.columns

# 9. Remove Instance Suffix (for Baseline)

In [ ]:
# pdf_renamed.columns = pdf_renamed.columns.str.replace(r'_i\d+$', '', regex=True)
# pdf_renamed.columns

# 10. Upload to DNAnexus

In [ ]:
# Save as CSV
pdf_renamed.to_csv('UKB_obesity_traits_all_raw_v1_20260205.csv', index=False)

In [ ]:
!pwd
!ls

In [ ]:
# Upload to DNAnexus
!dx upload 'UKB_obesity_traits_all_raw_v1_20260205.csv' -p --path '/Shiqi/metabolic traits/UKB_obesity_traits_all_raw_v1_20260205.csv' --brief